# These commands are helpful to see astroquery version and relevant steuptools


In [1]:
#import pkg_resources
#print(pkg_resources.__file__)
#pip install setuptools
#import sys
#print(sys.executable)
#print(sys.version)

In [2]:
#pip install astroquery

In [3]:
#pip install spectral_cube

In [4]:
#pip install pandas

# Import Libraries

In [4]:
from astroquery.alma import Alma
import os
import numpy as np
import pandas as pd
from astroquery.alma.utils import parse_frequency_support
import astropy.units as u
from astropy.table import Table
import shutil
from pathlib import Path
import tarfile
from astropy.io import fits
from spectral_cube import SpectralCube
import matplotlib.pyplot as plt 
#import alminer
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import add_beam
from glob import glob

# Edit this for relevant molecualr line

In [5]:
#Add sio rest frequency
sio_rest = 347.330581 * u.GHz #Splatalouge SiO (8-7)

#CO_rest = 345.7959899 * u.GHz #CO (3-2)

# These are the same keywords on the ALMA Archive: 1) Disks around low-mass stars, 2) Low mass star formation, 3) Outflows, jets, and ionized winds

In [6]:

# Give me all columns from the database. We are searching rowns with ALMA metadata
alma = Alma()
alma.archive_url = 'https://almascience.eso.org'

# Use ALMA AstroTAP query to find the dataproducts with observation keywords stored in ALMA data

In [19]:

# Check available columns in the ALMA TAP table
query_columns = """
SELECT TOP 1 *
FROM ivoa.obscore
"""


# Convert TAPResults to Astropy Table
column_table = column_results.to_table()

# Print all available column names
#print(column_table.colnames)



In [26]:

query = """
SELECT
    obs_publisher_did,
    member_ous_uid,
    target_name,
    frequency_support,
    science_keyword,
    antenna_arrays
FROM ivoa.obscore
WHERE dataproduct_type = 'cube'
AND (
      science_keyword LIKE '% jets and ionized winds%'
      OR science_keyword LIKE '%disks around low mass stars%'
  )
AND (
      antenna_arrays LIKE '%:DA%'
      OR antenna_arrays LIKE '%:DV%'
      OR antenna_arrays LIKE '%:PM%'
  )
AND antenna_arrays NOT LIKE '%:CM%'
"""

results = alma.query_tap(query)

print("Total 12m observations:", len(results))


Total 12m observations: 20624


# Parse the frequency support in the ALMA Data

In [20]:
#Freuency support stored as string so need to parse frequncy and only need frequncy ranges
frequency_support_str = results['frequency_support'][:1]
print(frequency_support_str)
print(type(frequency_support_str))

['[217.04..217.17GHz,141.11kHz,11.8mJy/beam@10km/s,2.8mJy/beam@native, XX YY] U [218.26..218.39GHz,141.11kHz,10.9mJy/beam@10km/s,2.6mJy/beam@native, XX YY] U [219.50..219.62GHz,141.11kHz,11.6mJy/beam@10km/s,2.8mJy/beam@native, XX YY] U [220.34..220.46GHz,141.11kHz,14.3mJy/beam@10km/s,3.5mJy/beam@native, XX YY] U [230.48..230.60GHz,141.11kHz,13.6mJy/beam@10km/s,3.4mJy/beam@native, XX YY] U [231.16..231.28GHz,141.11kHz,13mJy/beam@10km/s,3.2mJy/beam@native, XX YY] U [232.00..233.98GHz,31250.00kHz,11.1mJy/beam@10km/s,693.7uJy/beam@native, XX YY]']
<class 'numpy.ma.MaskedArray'>


In [21]:
#Parse the frequncy support
for a in range(len(frequency_support_str)):
    freq_ranges = parse_frequency_support(frequency_support_str[a])
    print(freq_ranges,a)
    

[[217.04 217.17]
 [218.26 218.39]
 [219.5  219.62]
 [220.34 220.46]
 [230.48 230.6 ]
 [231.16 231.28]
 [232.   233.98]] GHz 0


# Setup a frequency range to only find relevant data products with respect to the rest frequency

In [22]:
# Set up frequency range and in results find the range

matches = [] #Empty list
#print(type(results))
for ind,x in enumerate(results):
    #if ind >= 2:
        #continue
    ranges = parse_frequency_support(x['frequency_support'])
    #print(ranges)

    for fmin, fmax in ranges:
        if sio_rest >= fmin and sio_rest <= fmax:
            matches.append(x)
            break
print(len(matches))

958


In [27]:
#convert results to a table
table = results.to_table()

In [28]:
#print columnames
table.colnames

['obs_publisher_did',
 'member_ous_uid',
 'target_name',
 'frequency_support',
 'science_keyword',
 'antenna_arrays']

# Found Matches within the frequency range

In [29]:
# Convert list to a astropy.table
matches = Table(rows=matches, names=table.colnames)
#print(len(matches))

In [58]:
#print(matches)

In [41]:
#subset = 0 # only for 1 source
subset = matches[:500] #download all files

In [42]:
#print(len(subset))
#print(subset['member_ous_uid'])

# Expand the files infromation

In [43]:
#uid = matches['member_ous_uid'][subset] # only for 1 source
uid = subset['member_ous_uid']# 600 rows from the sample


products = alma.get_data_info (uid,expand_tarfiles=True)

#print(products)

# Only selecting tar file contain science products

In [44]:

science_tar = products[
    [('001_of_001.tar' in str(x)) for x in products['access_url']]
]

#print(len(science_tar))
#print(science_tar)

# Expand tar file and only select cube data products

In [45]:
cube_files = []

for row in products:
    if 'cube' in str(row).lower() and '.fits' in str(row).lower():
        cube_files.append(row)
        print(row)

for x in cube_files:
    print(len(x))

         ID                                                                       access_url                                                             service_def error_message semantics local_semantics                        description                           content_type   content_length content_qualifier link_auth link_authorized
                                                                                                                                                                                                                                                                                              byte                                                 
-------------------- ----------------------------------------------------------------------------------------------------------------------------------- ----------- ------------- --------- --------------- ---------------------------------------------------------- ---------------- -------------- ----------------- ------

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



# Download Data

In [46]:
alma.cache_location = '/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files' #save fits file to that location
download_dir = Path("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files")
download_dir.mkdir(exist_ok=True)

# Check available columns
#print(cube_files.colnames)

# Download selected files
downloaded_files = alma.download_files(
    [row['access_url'] for row in cube_files]
)
# Move files
for f in downloaded_files:
    shutil.move(f, download_dir / Path(f).name)

KeyboardInterrupt: 

In [47]:

from pathlib import Path

# Directory containing previously downloaded FITS files
download_dir = Path("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files")

# Find existing FITS files
downloaded_files = sorted(download_dir.glob("*.fits"))

print("Total existing FITS files:", len(downloaded_files))

for f in downloaded_files:
    print(f.name)


Total existing FITS files: 163
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw0_347327MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw1_349001MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw2_337393MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.IRAS_16293-2422_sci.spw3_335485MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw0_347329MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw1_349004MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw2_337395MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1517-2422_bp.spw3_335487MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw0_347328MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw1_349002MHz.12m.cube.I.pbcor.fits
member.uid___A001_X122_X4a1.ari_l.J1625-2527_ph.spw2_337394MHz.12m.cube.I.pbcor.fits
member.uid

In [48]:
print(len(downloaded_files))

163


In [37]:
#print(downloaded_files)

In [38]:
#print(download_dir.resolve())
#print(alma.cache_location)

In [39]:
#print(download_dir.exists())
#p = Path("/users/skaur/Machine-Learning-Project/Astromorph_project/alma_astroquery/alma_TAP/scratch_files")
#print(p.exists())

In [40]:
#Alma.help_tap()

# Get Rest Frequency for all fits files

In [49]:
#download_dir = Path('/users/skaur/lustre/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files')

for f in download_dir.glob("*.fits"):
    if "cube" in f.name.lower():
        header = fits.getheader(f)

        print("\n", f.name)
        print("Rest frequency:", header.get("RESTFRQ", "No RESTFRQ"))


 member.uid___A001_X3621_X98a.DHNb71_sci.spw21.cube.I.sd.fits
Rest frequency: 347330578600.0

 member.uid___A001_X1465_X1bc.Lh03_sci.spw27.cube.I.pbcor.fits
Rest frequency: 345650000000.0

 member.uid___A001_X2d20_Xebd._G206.93-16.61W2__sci.spw25.cube.I.pbcor.fits
Rest frequency: 346528481000.0

 member.uid___A001_X1465_X2b6b.HH_211_sci.spw29.cube.I.pbcor.fits
Rest frequency: 346998347000.0

 member.uid___A001_X3788_X5e32.10295-5746_sci.spw31.cube.I.selfcal.pbcor.fits
Rest frequency: 358605800000.0

 member.uid___A001_X133d_X3cbd.iras16293-2422_sci.spw17.cube.I.sd.fits
Rest frequency: 345795990000.0

 member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
Rest frequency: 347060000000.0

 member.uid___A001_X1465_X1c0.S09_sci.spw25.cube.I.pbcor.fits
Rest frequency: 344660000000.0

 member.uid___A001_X1465_X1c0.S03_sci.spw25.cube.I.pbcor.fits
Rest frequency: 344660000000.0

 member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits
Rest frequency: 346950000000.0


In [50]:
# Use rest frequency to find the right SiO cube FITS files
sio_files = []

for f in download_dir.glob("*.fits"):

    header = fits.getheader(f)

    restfreq = header.get("RESTFRQ", header.get("RESTFREQ", None))

    if restfreq is not None:

        freq1 = header["CRVAL3"] + (1 - header["CRPIX3"]) * header["CDELT3"]
        freq2 = header["CRVAL3"] + (header["NAXIS3"] - header["CRPIX3"]) * header["CDELT3"]

        freq_min_GHz = min(freq1, freq2) / 1e9
        freq_max_GHz = max(freq1, freq2) / 1e9

        if freq_min_GHz <= 347.33058 <= freq_max_GHz:
            sio_files.append(f)

In [51]:
print("SiO files:")

for f in sio_files:
    print(f)

print("Total SiO files:", len(sio_files))

SiO files:
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3621_X98a.DHNb71_sci.spw21.cube.I.sd.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X133d_X3cb7.iras16293-2422_sci.spw19.cube.I.sd.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh03_sci.spw29.cube.I.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S07_sci.spw29.cube.I.pbcor.fits
/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A00

In [54]:
import glob
all_files = glob.glob("/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/*.fits*")
#all_files
#print(Path(all_files[0]).name)

In [55]:
import re
from pathlib import Path

keep_files = []

# Automatically get observation IDs and SPWs from SiO files
sio_spws = set()

for f in sio_files:
    filename = Path(f).name.lower()

    match = re.search(r"(member\.uid.*?_sci)\.(spw\d+)", filename)

    if match:
        sio_spws.add((match.group(1), match.group(2)))

# Keep corresponding files
for var in all_files:
    filename = Path(var).name.lower()

    match = re.search(r"(member\.uid.*?_sci)\.(spw\d+)", filename)

    if match and (match.group(1), match.group(2)) in sio_spws:

        if (".pb." in filename or
            ".pbcor." in filename or
            ".mask." in filename or
            ("selfcal" in filename and "pbcor" in filename)):

            keep_files.append(var)

print("SiO observation and spectral windows:", sio_spws)
print("Total keep files:", len(keep_files))

SiO observation and spectral windows: {('member.uid___a001_x1465_x1c0.s04_sci', 'spw29'), ('member.uid___a001_x3621_x990.dhnb72_sci', 'spw21'), ('member.uid___a001_x1465_x1c0.s06_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s09_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s05_sci', 'spw29'), ('member.uid___a001_x3788_x5e32.10365-5803_sci', 'spw27'), ('member.uid___a001_x2d20_xec0._g210.37-19.53s__sci', 'spw29'), ('member.uid___a001_x133d_x3cb7.iras16293-2422_sci', 'spw19'), ('member.uid___a001_x1465_x1bc.lh01_sci', 'spw29'), ('member.uid___a001_x3621_x98a.dhnb71_sci', 'spw21'), ('member.uid___a001_x122_x4a1.ari_l.iras_16293-2422_sci', 'spw0'), ('member.uid___a001_x1465_x1c0.s10_sci', 'spw29'), ('member.uid___a001_x1465_x1bc.lh03_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s03_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s08_sci', 'spw29'), ('member.uid___a001_x1465_x1c0.s07_sci', 'spw29'), ('member.uid___a001_x3621_x996.dhnb73_sci', 'spw21'), ('member.uid___a001_x3788_x5e32.10

# Look all files in the download folder

In [32]:
#var = './alma_TAP/scratch_files/member.uid___A001_X3621_X98e.DHNb72_sci.spw20.cube.I.pbcor.fits'
#var = './'+str(sio_files)
#mask_file = var.split('pbcor.fits')[0]+'mask.fits.gz'
#pb_file = var.split('pbcor.fits')[0] +'pb.fits.gz'

In [33]:
#keep_files = []
#keep_files.append(var)
#keep_files.append(mask_file)
#keep_files.append(pb_file)
#print(len(keep_files))


In [59]:
keep_files_names = [Path(f).name for f in keep_files]

print("keep_files:")
for f in keep_files:
    print(repr(f))
    #print(len(f))


print("\nall_files:")
for f in all_files:
    print(repr(f))

keep_files:
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pb.fits.gz'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S08_sci.spw29.cube.I.mask.fits.gz'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh03_sci.spw29.cube.I.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S07_sci.spw29.cube.I.pbcor.fits'
'/scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.ui

## Move SiO Fits files in the final folder

In [60]:
for y in all_files:
    if Path(y).name in keep_files_names :
        destination = Path(y.replace('scratch_files', 'final_fits_file'))
        #print(destination)
        # Make sure destination directory exists
        destination.parent.mkdir(parents=True, exist_ok=True)

        shutil.move(Path(y), destination)
        print("Moved:", y)

Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pb.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S08_sci.spw29.cube.I.mask.fits.gz
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S03_sci.spw29.cube.I.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X3788_X5e32.10295-5746_sci.spw27.cube.I.selfcal.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1bc.Lh03_sci.spw29.cube.I.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP/scratch_files/member.uid___A001_X1465_X1c0.S07_sci.spw29.cube.I.pbcor.fits
Moved: /scratch/gsc9zr/Machine-Learning-Project/Astromorph_project/alma_TAP

## Delete Fits files have no SiO lines

In [61]:
remaining_files = glob.glob('./alma_TAP/scratch_files/*')

for y in remaining_files:
    #os.remove(y)
    print("Deleted:", y)
print(len(remaining_files))

0


# Open SiO fits files from the folder

In [45]:
# Folder containing the final FITS files
final_dir = Path("./alma_TAP/final_fits_file")

# Find the pbcor FITS file
pbcor_files = list(final_dir.glob("*pbcor.fits"))

#print("Found:", pbcor_files)
#print(len(pbcor_files))